# 🏃‍♂️ Sports Performance Tracker
### Advanced Training Analytics Platform

A comprehensive tool for tracking and analyzing athletic performance across multiple metrics including lap times, biometric data, environmental conditions, and equipment.

In [2]:
# Install required packages
# import sys
# !{sys.executable} -m pip install pandas numpy matplotlib seaborn plotly ipywidgets

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import json
import warnings
warnings.filterwarnings('ignore')

# Set styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All packages loaded successfully!")

/home/kceniabougrova/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


✅ All packages loaded successfully!


## 📊 Data Structure Setup

In [4]:
# Initialize data storage
class PerformanceTracker:
    def __init__(self):
        self.training_data = pd.DataFrame(columns=[
            'date', 'time', 'activity_type', 'distance_km', 'duration_min',
            'has_track', 'lap_times', 'avg_heart_rate', 'max_heart_rate',
            'avg_pace', 'calories', 'hrv', 'vo2_max', 'recovery_score',
            'temperature_c', 'humidity', 'elevation_gain_m', 'terrain_type',
            'shoe_brand', 'shoe_model', 'notes'
        ])
        
    def add_training(self, data):
        """Add a new training session"""
        new_row = pd.DataFrame([data])
        self.training_data = pd.concat([self.training_data, new_row], ignore_index=True)
        return True
    
    def get_data(self):
        """Return all training data"""
        return self.training_data
    
    def save_to_csv(self, filename='training_data.csv'):
        """Save data to CSV"""
        self.training_data.to_csv(filename, index=False)
        print(f"✅ Data saved to {filename}")
    
    def load_from_csv(self, filename='training_data.csv'):
        """Load data from CSV"""
        try:
            self.training_data = pd.read_csv(filename)
            print(f"✅ Data loaded from {filename}")
            return True
        except FileNotFoundError:
            print(f"⚠️ File {filename} not found. Starting fresh.")
            return False

# Initialize tracker
tracker = PerformanceTracker()
tracker.load_from_csv()

print("✅ Performance Tracker initialized!")

⚠️ File training_data.csv not found. Starting fresh.
✅ Performance Tracker initialized!


## 🎯 Training Entry GUI

In [5]:
# Create GUI widgets
style = {'description_width': '150px'}
layout = widgets.Layout(width='400px')

# Basic Info
date_input = widgets.DatePicker(
    description='Date:',
    value=datetime.now().date(),
    style=style,
    layout=layout
)

time_input = widgets.Text(
    description='Time:',
    value=datetime.now().strftime('%H:%M'),
    placeholder='HH:MM',
    style=style,
    layout=layout
)

activity_type = widgets.Dropdown(
    options=['Running', 'Cycling', 'Swimming', 'Interval Training', 'Tempo Run', 'Long Run', 'Recovery Run', 'Race'],
    description='Activity Type:',
    style=style,
    layout=layout
)

distance = widgets.FloatText(
    description='Distance (km):',
    value=0.0,
    style=style,
    layout=layout
)

duration = widgets.FloatText(
    description='Duration (min):',
    value=0.0,
    style=style,
    layout=layout
)

# Track-specific
has_track = widgets.Checkbox(
    description='Track Session',
    value=False,
    style=style
)

lap_times_input = widgets.Textarea(
    description='Lap Times (400m):',
    placeholder='Enter lap times separated by commas (e.g., 72, 71.5, 73, 70)',
    style=style,
    layout=widgets.Layout(width='400px', height='80px')
)

# Biometric Data
avg_hr = widgets.IntText(
    description='Avg Heart Rate:',
    value=0,
    style=style,
    layout=layout
)

max_hr = widgets.IntText(
    description='Max Heart Rate:',
    value=0,
    style=style,
    layout=layout
)

avg_pace = widgets.Text(
    description='Avg Pace (min/km):',
    placeholder='e.g., 5:30',
    style=style,
    layout=layout
)

calories = widgets.IntText(
    description='Calories:',
    value=0,
    style=style,
    layout=layout
)

hrv = widgets.FloatText(
    description='HRV (ms):',
    value=0.0,
    style=style,
    layout=layout
)

vo2_max = widgets.FloatText(
    description='VO2 Max:',
    value=0.0,
    style=style,
    layout=layout
)

recovery_score = widgets.IntSlider(
    description='Recovery Score:',
    min=0,
    max=100,
    value=70,
    style=style,
    layout=layout
)

# Environmental
temperature = widgets.FloatText(
    description='Temperature (°C):',
    value=20.0,
    style=style,
    layout=layout
)

humidity = widgets.IntSlider(
    description='Humidity (%):',
    min=0,
    max=100,
    value=50,
    style=style,
    layout=layout
)

elevation_gain = widgets.FloatText(
    description='Elevation Gain (m):',
    value=0.0,
    style=style,
    layout=layout
)

terrain = widgets.Dropdown(
    options=['Flat', 'Rolling', 'Hilly', 'Mountain', 'Track', 'Treadmill'],
    description='Terrain:',
    style=style,
    layout=layout
)

# Equipment
shoe_brand = widgets.Dropdown(
    options=['Nike', 'Brooks', 'Adidas', 'Asics', 'New Balance', 'Hoka', 'Saucony', 'On Running', 'Other'],
    description='Shoe Brand:',
    style=style,
    layout=layout
)

shoe_model = widgets.Text(
    description='Shoe Model:',
    placeholder='e.g., Vaporfly 3, Ghost 15',
    style=style,
    layout=layout
)

notes = widgets.Textarea(
    description='Notes:',
    placeholder='How did you feel? Any insights?',
    style=style,
    layout=widgets.Layout(width='400px', height='80px')
)

# Submit button
submit_button = widgets.Button(
    description='📝 Save Training',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

output_area = widgets.Output()

def on_submit_clicked(b):
    with output_area:
        clear_output()
        
        # Prepare data
        training_entry = {
            'date': str(date_input.value),
            'time': time_input.value,
            'activity_type': activity_type.value,
            'distance_km': distance.value,
            'duration_min': duration.value,
            'has_track': has_track.value,
            'lap_times': lap_times_input.value if has_track.value else '',
            'avg_heart_rate': avg_hr.value,
            'max_heart_rate': max_hr.value,
            'avg_pace': avg_pace.value,
            'calories': calories.value,
            'hrv': hrv.value,
            'vo2_max': vo2_max.value,
            'recovery_score': recovery_score.value,
            'temperature_c': temperature.value,
            'humidity': humidity.value,
            'elevation_gain_m': elevation_gain.value,
            'terrain_type': terrain.value,
            'shoe_brand': shoe_brand.value,
            'shoe_model': shoe_model.value,
            'notes': notes.value
        }
        
        tracker.add_training(training_entry)
        tracker.save_to_csv()
        
        print("✅ Training session saved successfully!")
        print(f"📊 Total sessions: {len(tracker.get_data())}")

submit_button.on_click(on_submit_clicked)

# Show/hide lap times based on track checkbox
def on_track_change(change):
    lap_times_input.layout.display = 'block' if change['new'] else 'none'

has_track.observe(on_track_change, names='value')
lap_times_input.layout.display = 'none'

print("✅ GUI widgets created!")

✅ GUI widgets created!


In [6]:
# Display the GUI
display(HTML("<h2 style='color: #2E86AB;'>📋 Training Entry Form</h2>"))

# Create organized sections
basic_info = widgets.VBox([
    widgets.HTML("<h3 style='color: #A23B72;'>Basic Information</h3>"),
    date_input, time_input, activity_type, distance, duration
])

track_info = widgets.VBox([
    widgets.HTML("<h3 style='color: #A23B72;'>Track Details</h3>"),
    has_track, lap_times_input
])

biometric_info = widgets.VBox([
    widgets.HTML("<h3 style='color: #A23B72;'>Biometric Data</h3>"),
    avg_hr, max_hr, avg_pace, calories, hrv, vo2_max, recovery_score
])

environmental_info = widgets.VBox([
    widgets.HTML("<h3 style='color: #A23B72;'>Environmental Conditions</h3>"),
    temperature, humidity, elevation_gain, terrain
])

equipment_info = widgets.VBox([
    widgets.HTML("<h3 style='color: #A23B72;'>Equipment</h3>"),
    shoe_brand, shoe_model
])

notes_section = widgets.VBox([
    widgets.HTML("<h3 style='color: #A23B72;'>Additional Notes</h3>"),
    notes
])

# Layout in columns
left_column = widgets.VBox([basic_info, track_info])
middle_column = widgets.VBox([biometric_info, equipment_info])
right_column = widgets.VBox([environmental_info, notes_section])

form_layout = widgets.HBox([left_column, middle_column, right_column])

display(form_layout)
display(submit_button)
display(output_area)

Button(button_style='success', description='📝 Save Training', layout=Layout(height='40px', width='200px'), sty…

Output()

## 📈 Performance Analytics

In [7]:
# View current data
data = tracker.get_data()

if len(data) > 0:
    display(HTML("<h3 style='color: #2E86AB;'>📊 Training History</h3>"))
    display(data.tail(10))
    
    # Basic statistics
    print(f"\n📈 Summary Statistics:")
    print(f"Total Sessions: {len(data)}")
    print(f"Total Distance: {data['distance_km'].sum():.2f} km")
    print(f"Total Duration: {data['duration_min'].sum():.2f} min ({data['duration_min'].sum()/60:.1f} hours)")
    print(f"Avg Heart Rate: {data['avg_heart_rate'].mean():.0f} bpm")
    print(f"Avg Recovery Score: {data['recovery_score'].mean():.0f}")
else:
    print("⚠️ No training data yet. Add some sessions using the form above!")

⚠️ No training data yet. Add some sessions using the form above!


## 📊 Visualization Dashboard

In [8]:
def create_visualizations():
    data = tracker.get_data()
    
    if len(data) == 0:
        print("⚠️ No data to visualize. Add training sessions first!")
        return
    
    # Prepare data
    data['date'] = pd.to_datetime(data['date'])
    data = data.sort_values('date')
    
    # 1. Training Volume Over Time
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(
        x=data['date'], 
        y=data['distance_km'],
        mode='lines+markers',
        name='Distance',
        line=dict(color='#2E86AB', width=2),
        marker=dict(size=8)
    ))
    fig1.update_layout(
        title='Training Volume Over Time',
        xaxis_title='Date',
        yaxis_title='Distance (km)',
        hovermode='x unified',
        template='plotly_white'
    )
    fig1.show()
    
    # 2. Heart Rate vs Recovery Score
    if data['avg_heart_rate'].sum() > 0:
        fig2 = px.scatter(
            data, 
            x='avg_heart_rate', 
            y='recovery_score',
            size='distance_km',
            color='activity_type',
            hover_data=['date', 'distance_km', 'duration_min'],
            title='Heart Rate vs Recovery Score'
        )
        fig2.update_layout(template='plotly_white')
        fig2.show()
    
    # 3. Performance by Shoe Brand
    if data['shoe_brand'].notna().any():
        shoe_stats = data.groupby('shoe_brand').agg({
            'distance_km': 'sum',
            'avg_heart_rate': 'mean',
            'recovery_score': 'mean'
        }).reset_index()
        
        fig3 = go.Figure()
        fig3.add_trace(go.Bar(
            x=shoe_stats['shoe_brand'],
            y=shoe_stats['distance_km'],
            name='Total Distance',
            marker_color='#F18F01'
        ))
        fig3.update_layout(
            title='Distance by Shoe Brand',
            xaxis_title='Shoe Brand',
            yaxis_title='Total Distance (km)',
            template='plotly_white'
        )
        fig3.show()
    
    # 4. Environmental Impact on Performance
    if data['temperature_c'].sum() > 0:
        fig4 = px.scatter(
            data,
            x='temperature_c',
            y='avg_heart_rate',
            color='terrain_type',
            size='distance_km',
            title='Temperature vs Heart Rate Performance',
            labels={'temperature_c': 'Temperature (°C)', 'avg_heart_rate': 'Avg Heart Rate (bpm)'}
        )
        fig4.update_layout(template='plotly_white')
        fig4.show()
    
    # 5. Track Performance (Lap Times)
    track_sessions = data[data['has_track'] == True]
    if len(track_sessions) > 0:
        print("\n🏃 Track Sessions Analysis:")
        for idx, row in track_sessions.iterrows():
            if row['lap_times']:
                laps = [float(x.strip()) for x in str(row['lap_times']).split(',') if x.strip()]
                if laps:
                    print(f"\n📅 {row['date'].strftime('%Y-%m-%d')} - {row['activity_type']}")
                    print(f"   Laps: {len(laps)} x 400m")
                    print(f"   Fastest: {min(laps):.1f}s | Slowest: {max(laps):.1f}s | Avg: {np.mean(laps):.1f}s")
                    print(f"   Consistency (std): {np.std(laps):.2f}s")

# Create visualization button
viz_button = widgets.Button(
    description='📊 Generate Visualizations',
    button_style='info',
    layout=widgets.Layout(width='250px', height='40px')
)

def on_viz_clicked(b):
    create_visualizations()

viz_button.on_click(on_viz_clicked)
display(viz_button)

Button(button_style='info', description='📊 Generate Visualizations', layout=Layout(height='40px', width='250px…

## 🎯 Advanced Analytics: Correlations & Insights

In [ ]:
def advanced_analytics():
    data = tracker.get_data()
    
    if len(data) < 5:
        print("⚠️ Need at least 5 sessions for meaningful analytics")
        return
    
    # Prepare numeric columns for correlation
    numeric_cols = ['distance_km', 'duration_min', 'avg_heart_rate', 'max_heart_rate',
                   'calories', 'hrv', 'vo2_max', 'recovery_score', 'temperature_c',
                   'humidity', 'elevation_gain_m']
    
    # Filter to available columns with data
    available_cols = [col for col in numeric_cols if col in data.columns and data[col].sum() != 0]
    
    if len(available_cols) > 2:
        # Correlation heatmap
        plt.figure(figsize=(12, 10))
        correlation_matrix = data[available_cols].corr()
        sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                   square=True, linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title('Performance Metrics Correlation Matrix', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # Key insights
        print("\n🔍 Key Correlations:")
        correlations = []
        for i in range(len(correlation_matrix.columns)):
            for j in range(i+1, len(correlation_matrix.columns)):
                corr_value = correlation_matrix.iloc[i, j]
                if abs(corr_value) > 0.5:
                    correlations.append((
                        correlation_matrix.columns[i],
                        correlation_matrix.columns[j],
                        corr_value
                    ))
        
        correlations.sort(key=lambda x: abs(x[2]), reverse=True)
        
        for var1, var2, corr in correlations[:5]:
            direction = "positively" if corr > 0 else "negatively"
            print(f"   • {var1} is {direction} correlated with {var2} (r={corr:.2f})")
    
    # Activity type breakdown
    print("\n📊 Activity Type Breakdown:")
    activity_stats = data.groupby('activity_type').agg({
        'distance_km': ['count', 'sum', 'mean'],
        'duration_min': 'mean',
        'avg_heart_rate': 'mean',
        'recovery_score': 'mean'
    }).round(2)
    print(activity_stats)
    
    # Shoe performance comparison
    if data['shoe_brand'].notna().sum() > 0:
        print("\n👟 Shoe Performance Comparison:")
        shoe_performance = data.groupby('shoe_brand').agg({
            'distance_km': 'sum',
            'avg_heart_rate': 'mean',
            'recovery_score': 'mean',
            'duration_min': 'mean'
        }).round(2)
        print(shoe_performance)

# Analytics button
analytics_button = widgets.Button(
    description='🔬 Advanced Analytics',
    button_style='warning',
    layout=widgets.Layout(width='250px', height='40px')
)

def on_analytics_clicked(b):
    advanced_analytics()

analytics_button.on_click(on_analytics_clicked)
display(analytics_button)

## 💾 Data Management

In [ ]:
# Export data
export_button = widgets.Button(
    description='💾 Export to CSV',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

def on_export_clicked(b):
    tracker.save_to_csv('my_training_data.csv')
    print("✅ Data exported to 'my_training_data.csv'")

export_button.on_click(on_export_clicked)

# Load data
load_button = widgets.Button(
    description='📂 Load from CSV',
    button_style='info',
    layout=widgets.Layout(width='200px')
)

def on_load_clicked(b):
    tracker.load_from_csv('my_training_data.csv')
    print(f"✅ Loaded {len(tracker.get_data())} training sessions")

load_button.on_click(on_load_clicked)

display(widgets.HBox([export_button, load_button]))

## 🎯 Sample Data (For Demo)

Run this cell to generate sample data for testing the visualizations:

In [ ]:
def generate_sample_data():
    """Generate realistic sample training data"""
    np.random.seed(42)
    
    activities = ['Running', 'Tempo Run', 'Long Run', 'Interval Training', 'Recovery Run']
    terrains = ['Flat', 'Rolling', 'Hilly', 'Track']
    shoes = [('Nike', 'Vaporfly 3'), ('Brooks', 'Ghost 15'), ('Adidas', 'Adizero Boston')]
    
    sample_sessions = []
    start_date = datetime.now() - timedelta(days=60)
    
    for i in range(30):
        session_date = start_date + timedelta(days=i*2)
        activity = np.random.choice(activities)
        
        # Generate realistic values based on activity type
        if activity == 'Long Run':
            distance = np.random.uniform(15, 25)
            duration = distance * np.random.uniform(5.5, 6.5)
            avg_hr = np.random.randint(135, 155)
        elif activity == 'Interval Training':
            distance = np.random.uniform(8, 12)
            duration = distance * np.random.uniform(4.5, 5.5)
            avg_hr = np.random.randint(160, 180)
        elif activity == 'Recovery Run':
            distance = np.random.uniform(5, 8)
            duration = distance * np.random.uniform(6.0, 7.0)
            avg_hr = np.random.randint(120, 140)
        else:
            distance = np.random.uniform(8, 15)
            duration = distance * np.random.uniform(5.0, 6.0)
            avg_hr = np.random.randint(145, 165)
        
        shoe_brand, shoe_model = shoes[i % len(shoes)]
        
        session = {
            'date': session_date.strftime('%Y-%m-%d'),
            'time': f"{np.random.randint(6, 19)}:{np.random.choice(['00', '30'])}",
            'activity_type': activity,
            'distance_km': round(distance, 2),
            'duration_min': round(duration, 1),
            'has_track': activity == 'Interval Training' and np.random.random() > 0.5,
            'lap_times': ','.join([str(round(np.random.uniform(68, 76), 1)) for _ in range(8)]) if activity == 'Interval Training' else '',
            'avg_heart_rate': avg_hr,
            'max_heart_rate': avg_hr + np.random.randint(15, 30),
            'avg_pace': f"{int(duration/distance)}:{int((duration/distance % 1) * 60):02d}",
            'calories': int(distance * 65),
            'hrv': round(np.random.uniform(45, 75), 1),
            'vo2_max': round(np.random.uniform(52, 62), 1),
            'recovery_score': np.random.randint(60, 95),
            'temperature_c': round(np.random.uniform(10, 28), 1),
            'humidity': np.random.randint(30, 80),
            'elevation_gain_m': round(np.random.uniform(0, 300), 0),
            'terrain_type': np.random.choice(terrains),
            'shoe_brand': shoe_brand,
            'shoe_model': shoe_model,
            'notes': np.random.choice(['Felt strong', 'Legs tired', 'Good pace', 'Hot day', ''])
        }
        
        tracker.add_training(session)
    
    tracker.save_to_csv()
    print(f"✅ Generated {len(sample_sessions)} sample training sessions!")
    print(f"📊 Total sessions in database: {len(tracker.get_data())}")

# Button to generate sample data
sample_button = widgets.Button(
    description='🎲 Generate Sample Data',
    button_style='warning',
    layout=widgets.Layout(width='250px', height='40px')
)

def on_sample_clicked(b):
    generate_sample_data()

sample_button.on_click(on_sample_clicked)
display(sample_button)

---

## 📝 Usage Instructions

### Quick Start:
1. **Add Training Data**: Fill out the form above and click "Save Training"
2. **Generate Visualizations**: Click the "Generate Visualizations" button
3. **View Analytics**: Click "Advanced Analytics" for deeper insights
4. **Export Data**: Use "Export to CSV" to save your data

### Features:
- ✅ Track training sessions with comprehensive metrics
- ✅ Special support for track workouts with lap times
- ✅ Biometric data tracking (HR, HRV, VO2 Max, Recovery)
- ✅ Environmental condition logging (temperature, humidity, elevation)
- ✅ Equipment tracking (shoe brands and models)
- ✅ Interactive visualizations and correlations
- ✅ Performance comparison across variables

### Perfect for Portfolio:
This notebook demonstrates:
- Data collection and management
- Interactive GUI development
- Sports performance analytics
- Data visualization skills
- Understanding of athletic training principles

---

**Created for roles in Sports Performance & Data Analytics**

*Target Companies: Brooks, Nike, Whoop, Strava, Garmin, Polar*